# Train a RoBERTa Classifier on the (preprocessed) HC3 Data

Note: this code was tested in Google Colab using an A100 GPU, and has not been verified in any other scenarios

In [25]:
import os

try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
  hc3_roberta_dir = "/content/drive/MyDrive/Colab Notebooks/NLU_project/hc3-roberta-classifier" # Directory to store intermediate and final versions of the RoBERTa model trained on HC3 data
except:
  pass

# Set up directory to store intermediate and final versions of the RoBERTa model trained on HC3 data
# Use Google Drive if available, otherwise save locally
if os.path.exists('/content/drive/MyDrive/Colab Notebooks/NLU_project/'):
  hc3_roberta_dir = "/content/drive/MyDrive/Colab Notebooks/NLU_project/hc3-roberta-classifier"
else:
  hc3_roberta_dir = "./hc3-roberta-classifier"

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

# Download the data from GitHub
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/HC3_Dataset/train.csv
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/HC3_Dataset/test.csv
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/HC3_Dataset/val.csv

# load in the HC3 data
hc3_dataset = load_dataset("csv", data_files={
    "train":      "train.csv",
    "test":       "test.csv",
    "val":        "val.csv",
})

In [27]:
from transformers import RobertaTokenizer

hc3_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return hc3_tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

hc3_tokenized_data = hc3_dataset.map(tokenize, batched=True)
hc3_tokenized_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/8471 [00:00<?, ? examples/s]

In [28]:
from transformers import RobertaForSequenceClassification

hc3_roberta = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Set up method to use when computing metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

# Set up training arguments
training_args = TrainingArguments(
    output_dir=hc3_roberta_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    # warmup_ratio=0.06,
    warmup_steps=250,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trnr = Trainer(
    model=hc3_roberta,
    args=training_args,
    train_dataset=hc3_tokenized_data["train"],
    eval_dataset=hc3_tokenized_data["val"],
    compute_metrics=compute_metrics,
)

trnr.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.007374,0.014091,0.997174,0.997176
2,0.004042,0.038554,0.994584,0.994596
3,0.001107,0.050972,0.993525,0.993542


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=12096, training_loss=0.017938512071647397, metrics={'train_runtime': 869.7334, 'train_samples_per_second': 222.506, 'train_steps_per_second': 13.908, 'total_flos': 5.091751454432256e+16, 'train_loss': 0.017938512071647397, 'epoch': 3.0})

In [30]:
# Evaluate on test set after training
trnr.evaluate(hc3_tokenized_data["test"])

{'eval_loss': 0.009834593161940575,
 'eval_accuracy': 0.9981112029276354,
 'eval_f1': 0.9981121550618278,
 'eval_runtime': 8.877,
 'eval_samples_per_second': 954.265,
 'eval_steps_per_second': 29.852,
 'epoch': 3.0}

In [31]:
# Save the model and tokenizer to avoid needing to train again

hc3_roberta.save_pretrained(hc3_roberta_dir)
hc3_tokenizer.save_pretrained(hc3_roberta_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/NLU_project/hc3-roberta-classifier/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/NLU_project/hc3-roberta-classifier/tokenizer.json')